In [ ]:
# Quick test for six-label-to-three-risk BERT on LIAR test set


import os
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# LIAR data is here
project_root = "/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1"
liar_data_dir = os.path.join(project_root, "data", "LIAR")

# Model copy is here
model_project_root = "/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/backend/fake_news_text_feature"
model_path = os.path.join(
    project_root,
    "model_training",
    "model_copy",
    "liar_6label_to_3risk_selected_model.pt"
)


print("LIAR data folder:", liar_data_dir)
print("Model path:", model_path)
print("Model exists:", os.path.exists(model_path))

base_model_name = "bert-base-uncased"
max_sequence_length = 128
batch_size = 32

six_label_names = [
    "pants-fire",
    "false",
    "barely-true",
    "half-true",
    "mostly-true",
    "true"
]

risk_label_names = [
    "low_risk",
    "medium_risk",
    "high_risk"
]

six_label_map = {
    label_name: label_index
    for label_index, label_name in enumerate(six_label_names)
}

six_to_risk_map = {
    "true": 0,
    "mostly-true": 0,
    "half-true": 1,
    "barely-true": 1,
    "false": 2,
    "pants-fire": 2
}

liar_column_names = [
    "id", "label", "statement", "subject", "speaker", "speaker_job",
    "state", "party", "barely_true_counts", "false_counts",
    "half_true_counts", "mostly_true_counts", "pants_on_fire_counts",
    "context"
]

test_file_path = os.path.join(liar_data_dir, "test.tsv")

test_df = pd.read_csv(
    test_file_path,
    sep="\t",
    header=None,
    names=liar_column_names
)

test_df = test_df[["label", "statement"]].copy()
test_df["statement"] = test_df["statement"].astype(str)
test_df["six_label"] = test_df["label"].map(six_label_map)
test_df["risk_label"] = test_df["label"].map(six_to_risk_map)
test_df = test_df.dropna(subset=["six_label", "risk_label"]).copy()
test_df["six_label"] = test_df["six_label"].astype(int)
test_df["risk_label"] = test_df["risk_label"].astype(int)

print("Test samples:", len(test_df))
print(test_df["risk_label"].value_counts().sort_index().rename(index={0: "low_risk", 1: "medium_risk", 2: "high_risk"}))

tokenizer = BertTokenizer.from_pretrained(base_model_name)

class LiarTestDataset(Dataset):
    def __init__(self, dataframe):
        self.statements = dataframe["statement"].tolist()
        self.risk_labels = dataframe["risk_label"].tolist()

    def __len__(self):
        return len(self.statements)

    def __getitem__(self, index):
        encoded_text = tokenizer(
            self.statements[index],
            truncation=True,
            padding="max_length",
            max_length=max_sequence_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded_text["input_ids"].squeeze(0),
            "attention_mask": encoded_text["attention_mask"].squeeze(0),
            "risk_label": torch.tensor(self.risk_labels[index], dtype=torch.long)
        }

test_loader = DataLoader(
    LiarTestDataset(test_df),
    batch_size=batch_size,
    shuffle=False
)

model = BertForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=6
)

checkpoint = torch.load(model_path, map_location=device)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()

true_risk_labels = []
predicted_risk_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        six_label_probabilities = torch.softmax(outputs.logits, dim=1)

        # Convert six LIAR probabilities into three product risk probabilities.
        high_risk_probability = (
            six_label_probabilities[:, six_label_names.index("pants-fire")] +
            six_label_probabilities[:, six_label_names.index("false")]
        )

        medium_risk_probability = (
            six_label_probabilities[:, six_label_names.index("barely-true")] +
            six_label_probabilities[:, six_label_names.index("half-true")]
        )

        low_risk_probability = (
            six_label_probabilities[:, six_label_names.index("mostly-true")] +
            six_label_probabilities[:, six_label_names.index("true")]
        )

        risk_probabilities = torch.stack(
            [
                low_risk_probability,
                medium_risk_probability,
                high_risk_probability
            ],
            dim=1
        )

        risk_predictions = torch.argmax(risk_probabilities, dim=1)

        true_risk_labels.extend(batch["risk_label"].cpu().numpy())
        predicted_risk_labels.extend(risk_predictions.cpu().numpy())

accuracy = accuracy_score(true_risk_labels, predicted_risk_labels)
macro_f1 = f1_score(true_risk_labels, predicted_risk_labels, average="macro")
matrix = confusion_matrix(
    true_risk_labels,
    predicted_risk_labels,
    labels=[0, 1, 2]
)

matrix_df = pd.DataFrame(
    matrix,
    index=["true_low", "true_medium", "true_high"],
    columns=["pred_low", "pred_medium", "pred_high"]
)

print("\nQuick test result")
print("=" * 50)
print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))
print("\nConfusion matrix:")
display(matrix_df)


Device: cuda
LIAR data folder: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/data/LIAR
Model path: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/model_copy/liar_6label_to_3risk_selected_model.pt
Model exists: True
Test samples: 1267
risk_label
low_risk       449
medium_risk    477
high_risk      341
Name: count, dtype: int64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Quick test result
Accuracy: 0.4846
Macro F1: 0.4743

Confusion matrix:


,pred_low,pred_medium,pred_high
true_low,285,123,41
true_medium,184,203,90
true_high,112,103,126
